In [1]:
# Husayn El Sharif
comment = """
This notebook imports the feature engineered data for clustering AMI data using k-means clustering.
"""

In [2]:
# import libraries
import numpy as np
import pandas as pd
import os

# sklearn for k-means clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# plotting
import plotly.express as px
import plotly.graph_objects as go

In [3]:
# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [4]:
# import data

input_features_path = "../cleaned_data/meter_engineered_features.csv"

# Load engineered features
df_feat = pd.read_csv(input_features_path)

print("Shape:", df_feat.shape)
df_feat.head()

Shape: (46, 27)


,meter,n_records,kwh_mean,kwh_std,kwh_cv,kwh_p95,kwh_max,v_mean,v_std,v_cv,...,f_pct_outside_band,i_mean,i_p95,i_max,v_i_corr,kwh_v_corr,kwh_ramp_mean,kwh_ramp_p95,v_ramp_mean,v_ramp_p95
0,BR02,7248,0.095011,0.079462,0.836345,0.23765,0.727,236.810189,52.307234,0.220883,...,0.141142,0.913232,2.37300,6.222,0.158479,0.191054,0.034669,0.1590,11.642128,75.9130
1,BR03,6576,0.131199,0.094349,0.719128,0.29800,0.363,232.355339,54.595009,0.234963,...,0.148571,1.255726,2.79800,3.427,0.256470,0.273988,0.020789,0.0830,12.209766,77.2534
2,BR04,7584,0.207436,0.177531,0.855836,0.51000,1.608,238.086277,40.498318,0.170099,...,0.106672,1.978553,4.68925,13.457,0.046916,0.056466,0.073649,0.3060,8.811777,52.1714
3,BR05,5476,0.054964,0.042352,0.770536,0.13425,0.320,235.024342,47.687209,0.202903,...,0.124361,0.603433,1.64425,3.554,0.160096,0.189616,0.019955,0.0710,10.037751,68.7205
4,BR06,7488,0.309622,0.280565,0.906154,0.92500,2.394,242.096807,40.045844,0.165413,...,0.107772,2.822769,7.64095,18.638,0.129542,0.125267,0.154531,0.5947,8.716384,51.9456


In [5]:
# Columns to exclude from clustering
exclude_cols = set(["meter"])

# Keep numeric feature columns only (k-means requires numeric)
numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

# Optionally exclude raw count metadata if present
if "n_records" in feature_cols:
    feature_cols.remove("n_records")

print("Number of clustering features:", len(feature_cols))


Number of clustering features: 25


In [6]:
# Handle missing values (k-means cannot accept NaNs)
X = df_feat[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan)

na_rate = X.isna().mean().sort_values(ascending=False)
print("Top missing-rate features:")
print(na_rate.head(10))

X = X.fillna(X.median(numeric_only=True))
X.describe().T.head(20)

Top missing-rate features:
kwh_mean    0.0
kwh_std     0.0
kwh_cv      0.0
kwh_p95     0.0
kwh_max     0.0
v_mean      0.0
v_std       0.0
v_cv        0.0
v_min       0.0
v_max       0.0
dtype: float64


,count,mean,std,min,25%,50%,75%,max
kwh_mean,46.0,0.141638,0.092126,0.007201,0.059331,0.134322,0.203602,0.321252
kwh_std,46.0,0.132716,0.087619,0.018476,0.064903,0.108332,0.177719,0.332687
kwh_cv,46.0,1.096371,0.767556,0.497613,0.782033,0.900616,1.063397,5.564492
kwh_p95,46.0,0.404372,0.301213,0.000000,0.183500,0.296500,0.534750,1.084000
kwh_max,46.0,1.001457,0.586741,0.176000,0.576500,0.951500,1.275500,2.394000
v_mean,46.0,230.362032,14.124000,201.031842,223.223358,232.516723,240.460502,253.313381
v_std,46.0,49.644468,17.660187,28.959536,36.080872,40.272081,65.537242,83.379834
v_cv,46.0,0.219891,0.090628,0.123010,0.148743,0.169641,0.301376,0.404747
v_min,46.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
v_max,46.0,291.031804,19.838269,250.541000,272.808250,291.123500,308.993250,327.609000


In [9]:
# Scale features: Because k-means uses Euclidean distance, we standardize features.

scaler = StandardScaler() # default: mean=0, std=1
X_scaled = scaler.fit_transform(X) # scale features, returns numpy array

X_scaled.shape

(46, 25)

In [10]:
# select k based on elbow method and silhouette score
comment = """
evaluate candidate values of *k* to balance:
- cluster compactness (inertia)
- cluster separation (silhouette)
- interpretability (utilities often prefer ~3–6 clusters)

The K-Means silhouette score is a metric ranging from -1 to +1 
that evaluates clustering quality by measuring how similar a point is to its own cluster (cohesion) 
versus other clusters (separation). A score closer to +1 indicates well-clustered data, 
while values near 0 or -1 indicate overlapping or incorrect assignments. 

Common Interpretation of Scores:
0.71 to 1.0: Strong clustering structure
0.51 to 0.70: Reasonable structure
0.26 to 0.50: Weak structure, may need more clusters
0.00 to 0.25: No substantial structure
-1.0 to -0.01: Incorrect clustering

This method helps confirm that clusters are dense and well-separated, 
serving as an alternative or complement to the Elbow Method. 
"""

k_range = range(2, 11)

inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=30)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

In [11]:
# --- Elbow Plot ---
fig_elbow = go.Figure()

fig_elbow.add_trace(
    go.Scatter(
        x=list(k_range),
        y=inertias,
        mode="lines+markers",
        name="Inertia"
    )
)

fig_elbow.update_layout(
    title="Elbow Method for K-Means",
    xaxis_title="Number of clusters (k)",
    yaxis_title="Inertia",
    template="plotly_white"
)

fig_elbow.show()

# --- Silhouette Plot ---
fig_sil = go.Figure()

fig_sil.add_trace(
    go.Scatter(
        x=list(k_range),
        y=sil_scores,
        mode="lines+markers",
        name="Silhouette Score"
    )
)

fig_sil.update_layout(
    title="Silhouette Analysis for K-Means",
    xaxis_title="Number of clusters (k)",
    yaxis_title="Silhouette Score",
    template="plotly_white"
)

fig_sil.show()

In [12]:
# appears that k=4 is a good balance between inertia and silhouette score

k_final = 4

fig_elbow.add_vline(x=k_final, line_dash="dash")
fig_sil.add_vline(x=k_final, line_dash="dash")


In [13]:
# Comments
comment = """
The silhouette scores around 0.30 to 0.35 indicate moderate cluster separation, 
which is expected for real-world AMI
"""

In [15]:
# final k means with k=k_final

inertias_final = []
sil_scores_final = []

km_final = KMeans(n_clusters=k_final, random_state=RANDOM_SEED, n_init=30)
labels_final = km_final.fit_predict(X_scaled)
inertias_final.append(km_final.inertia_)
sil_scores_final.append(silhouette_score(X_scaled, labels_final))

In [22]:
# Attach cluster labels to original data
df_clusters = df_feat.copy()
df_clusters['kmeans_cluster'] = labels_final
df_clusters[["meter", "kmeans_cluster"]]


,meter,kmeans_cluster
0,BR02,0
1,BR03,0
2,BR04,3
3,BR05,0
4,BR06,3
5,BR07,0
6,BR08,0
7,BR09,0
8,BR10,0
9,BR11,0


In [23]:
# How many meters are in each cluster?
cluster_sizes = (
    df_clusters["kmeans_cluster"]
    .value_counts()
    .sort_index()
    .rename("n_meters")
)

cluster_sizes


kmeans_cluster
0    24
1     8
2     6
3     8
Name: n_meters, dtype: int64

In [ ]:
# Cluster centroids (mean feature values per cluster)
cluster_centroids = (
    df_clusters
    .groupby("kmeans_cluster")[feature_cols]
    .mean()
)

cluster_centroids


,kwh_mean,kwh_std,kwh_cv,kwh_p95,kwh_max,v_mean,v_std,v_cv,v_min,v_max,...,f_pct_outside_band,i_mean,i_p95,i_max,v_i_corr,kwh_v_corr,kwh_ramp_mean,kwh_ramp_p95,v_ramp_mean,v_ramp_p95
kmeans_cluster,,,,,,,,,,,,,,,,,,,,,
0,0.106372,0.097391,0.996698,0.287446,0.810500,237.638444,39.689357,0.167473,0.0,292.249375,...,0.121684,1.018600,2.588646,7.614917,0.063619,0.085942,0.030980,0.132837,10.193011,61.610954
1,0.050426,0.061627,1.710727,0.138250,0.546875,215.130248,71.745399,0.333731,0.0,286.107500,...,0.237175,0.483292,1.308062,5.095250,0.202621,0.231122,0.015630,0.066750,24.777047,149.544294
2,0.213774,0.194351,0.895624,0.617442,1.453833,213.339239,77.839022,0.366382,0.0,307.582167,...,0.269068,2.009248,5.683067,13.253167,0.296647,0.323136,0.080172,0.336667,27.030042,158.657683
3,0.284548,0.263553,0.931598,0.861469,1.689625,236.531677,36.262954,0.153435,0.0,279.890625,...,0.107402,2.594559,7.514137,14.645125,0.037024,0.045243,0.105887,0.451000,8.678340,50.581925


In [26]:
# Normalize centroids for easier comparison
comment = """
Interpretation rule:
Positive → above-average
Negative → below-average
"""

centroids_z = (
    cluster_centroids
    - cluster_centroids.mean()
) / cluster_centroids.std()

centroids_z


,kwh_mean,kwh_std,kwh_cv,kwh_p95,kwh_max,v_mean,v_std,v_cv,v_min,v_max,...,f_pct_outside_band,i_mean,i_p95,i_max,v_i_corr,kwh_v_corr,kwh_ramp_mean,kwh_ramp_p95,v_ramp_mean,v_ramp_p95
kmeans_cluster,,,,,,,,,,,,,,,,,,,,,
0,-0.545469,-0.618121,-0.353932,-0.579405,-0.587762,0.906051,-0.778440,-0.794954,NaN,0.066691,...,-0.764713,-0.533319,-0.594180,-0.558953,-0.709068,-0.662917,-0.646218,-0.639887,-0.781082,-0.764504
1,-1.077059,-1.007053,1.491211,-1.037498,-1.080119,-0.796458,0.716257,0.710678,NaN,-0.450519,...,0.656360,-1.095500,-1.045796,-1.114044,0.432242,0.463792,-1.011080,-1.010917,0.742515,0.781342
2,0.475028,0.436304,-0.615120,0.433819,0.613754,-0.931929,1.000388,1.006357,NaN,1.357873,...,1.048796,0.507062,0.497112,0.683172,1.204263,1.177897,0.523032,0.504456,0.977886,0.941553
3,1.147500,1.188869,-0.522159,1.183084,1.054128,0.822336,-0.938205,-0.922080,NaN,-0.974045,...,-0.940443,1.121757,1.142865,0.989825,-0.927438,-0.978773,1.134266,1.146349,-0.939320,-0.958392


In [33]:
# Focus on features that matter most for interpretation
key_features = [
    "kwh_mean",
    "kwh_cv",
    "kwh_ramp_mean",
    "kwh_ramp_p95",
    "v_std",
    "v_pct_outside_band",
    "v_ramp_mean",
    "f_std",
    "f_pct_outside_band",
    "i_p95",
]

cluster_centroids[key_features].round(3)

,kwh_mean,kwh_cv,kwh_ramp_mean,kwh_ramp_p95,v_std,v_pct_outside_band,v_ramp_mean,f_std,f_pct_outside_band,i_p95
kmeans_cluster,,,,,,,,,,
0,0.106,0.997,0.031,0.133,39.689,0.380,10.193,7.760,0.122,2.589
1,0.050,1.711,0.016,0.067,71.745,0.619,24.777,14.210,0.237,1.308
2,0.214,0.896,0.080,0.337,77.839,0.581,27.030,15.311,0.269,5.683
3,0.285,0.932,0.106,0.451,36.263,0.325,8.678,7.181,0.107,7.514


In [34]:
centroids_z[key_features].round(3)

,kwh_mean,kwh_cv,kwh_ramp_mean,kwh_ramp_p95,v_std,v_pct_outside_band,v_ramp_mean,f_std,f_pct_outside_band,i_p95
kmeans_cluster,,,,,,,,,,
0,-0.545,-0.354,-0.646,-0.640,-0.778,-0.662,-0.781,-0.791,-0.765,-0.594
1,-1.077,1.491,-1.011,-1.011,0.716,0.981,0.743,0.730,0.656,-1.046
2,0.475,-0.615,0.523,0.504,1.000,0.720,0.978,0.990,1.049,0.497
3,1.147,-0.522,1.134,1.146,-0.938,-1.039,-0.939,-0.928,-0.940,1.143


In [35]:
# Export centroids_z to csv
output_centroids_z_path = "../results/kmeans_cluster_centroids_z.csv"
centroids_z.to_csv(output_centroids_z_path, index_label="kmeans_cluster")
print(f"Exported centroids_z to: {output_centroids_z_path}")

Exported centroids_z to: ../results/kmeans_cluster_centroids_z.csv


# Cluster-by-Cluster Interpretation 

# Summary Table:

| Cluster | Label                  | Primary Use                |
| ------- | ---------------------- | -------------------------- |
| 0       | Stable Baseline        | Reference / low priority   |
| 1       | Demand-Variable        | Usage characterization     |
| 2       | Power-Quality Volatile | Reliability monitoring / High Priority    |
| 3       | High-Load Ramping      | Capacity & stress planning |

## Cluster 0 — **Stable, Low-Load Baseline Meters**

**Strong signals**

* Below-average load and variability
  * `kwh_mean` −0.55
  * `kwh_std` −0.62
  * `kwh_cv` −0.35

* Very stable voltage and frequency
  * `v_std` −0.78
  * `f_pct_outside_band` −0.76

* Low ramping behavior
  * `kwh_ramp_mean` −0.65
  * `v_ramp_mean` −0.78

**Interpretation**
Meters in this cluster exhibit **consistently low, smooth consumption** with **stable voltage and frequency behavior**.

**Operational meaning**
* Represents a *healthy baseline population*
* Low operational risk
* Useful as a control/reference group

> *This cluster reflects normal, well-behaved AMI operation.*

## Cluster 1 — **Low-Load but Highly Variable (Demand-Driven) Meters**

**Strong signals**

* Very low average load
  * `kwh_mean` −1.08

* **High relative variability**
  * `kwh_cv` +1.49

* Elevated voltage & frequency excursions
  * `v_std` +0.72
  * `f_pct_outside_band` +0.66

* Low absolute current
  * `i_mean` −1.10

**Interpretation**
These meters consume **little energy overall**, but exhibit **erratic relative behavior**, likely due to intermittent or irregular usage patterns.

**Operational meaning**
* Variability likely driven by **customer behavior**, not asset stress
* May correspond to seasonal, intermittent, or low-usage customers
* Not a primary outage-risk group

> *High variability here is likely demand-driven rather than infrastructure-driven.*


## Cluster 2 — **Voltage- and Frequency-Volatile (Infrastructure-Stressed) Meters**

**Strong signals**

* Elevated voltage instability
  * `v_std` +1.00
  * `v_cv` +1.01

* Highest frequency excursion rate
  * `f_pct_outside_band` +1.05

* Above-average electrical stress
  * `i_mean` +0.51
  * `i_p95` +0.50

* Strong voltage–current and load–voltage coupling
  * `v_i_corr` +1.20
  * `kwh_v_corr` +1.18

**Interpretation**
Meters in this cluster show **clear power-quality instability**, strongly tied to electrical loading and upstream conditions. 

Meters indicating both high electrical load and high voltage simultaneously is a significant cause for concern for a power company and should be reported immediately. This combination indicates an unstable, high-power condition that risks damaging customer electronics, causing overheating, or starting electrical fires. 

If a cluster of meters shows this behavior, it is a high-priority, critical issue usually indicating a major distribution failure, such as a broken neutral wire or a failed transformer, requiring immediate utility intervention to prevent widespread equipment failure and safety hazards. 

**Operational meaning**
* High-priority group for **feeder / transformer investigation**
* Strong candidate cluster for proactive monitoring
* Closely aligned with **grid-driven reliability risk**

> *This cluster most directly reflects infrastructure-level stress signals.*

## Cluster 3 — **High-Load, High-Ramping Meters (Stress-Amplifying Customers)**

**Strong signals**

* Highest load and absolute variability
  * `kwh_mean` +1.15
  * `kwh_std` +1.19

* Extreme ramping behavior
  * `kwh_ramp_mean` +1.13
  * `kwh_ramp_p95` +1.15

* Highest current stress
  * `i_mean` +1.12
  * `i_p95` +1.14

* Surprisingly stable voltage & frequency
  * `v_std` −0.94
  * `f_pct_outside_band` −0.94

**Interpretation**
These meters impose **high and rapidly changing demand**, but without corresponding voltage or frequency instability.

**Operational meaning**
* Heavy consumers that may **amplify stress during events**
* Important for capacity planning and load management
* Not inherently unstable, but influential under peak conditions

> *This cluster captures load-driven stress rather than power-quality failure.*

# Executive Summary

K-means clustering of engineered AMI features segmented meters into four interpretable behavioral groups: 
  * stable low-load baseline meters
  * low-load but demand-variable meters
  * voltage- and frequency-volatile meters indicative of upstream infrastructure stress
  * high-load high-ramping meters that amplify system stress without exhibiting instability themselves. 
  
  Although cluster separation is moderate (reflecting the continuous nature of real AMI behavior), the resulting segments align cleanly with common utility monitoring and reliability workflows.


In [45]:
# Visualize cluster centroids as heatmap
# centroids_z is a DataFrame.
# rows = clusters, columns = features, values = z-scores


# Pick the most story-relevant features (edit as you like)
heatmap_features = [
    "kwh_mean", "kwh_cv", "kwh_ramp_mean", "kwh_ramp_p95",
    "v_mean", "v_std", "v_ramp_mean",
    "f_pct_outside_band",
    "i_mean", "i_p95",
    "v_i_corr", "kwh_v_corr"
]

feature_label_map = {
    "kwh_mean": "Mean Energy Consumption (kWh)",
    "kwh_cv": "Energy Variability (CV)",
    "kwh_ramp_mean": "Mean Energy Ramp Rate",
    "kwh_ramp_p95": "95th Percentile Energy Ramp",
    "v_mean": "Mean Voltage",
    "v_std": "Voltage Variability (Std)",
    "v_ramp_mean": "Mean Voltage Ramp",
    "f_pct_outside_band": "Frequency Excursions (%)",
    "i_mean": "Mean Current",
    "i_p95": "95th Percentile Current",
    "v_i_corr": "Voltage–Current Correlation",
    "kwh_v_corr": "Energy–Voltage Correlation"
}

z = centroids_z[heatmap_features]

# Map feature names to descriptive labels
x_labels = [feature_label_map[f] for f in heatmap_features]

fig = px.imshow(
    z,
    labels=dict(x="Feature", y="Cluster", color="Z-score"),
    x=x_labels,
    y=[f"Cluster {c}" for c in z.index],
    text_auto=".2f",
    aspect="auto",
    title="Cluster Feature Heatmap (Normalized Feature Centroids)"
)

fig.update_layout(template="plotly_white")
fig.update_xaxes(tickangle=-30)
fig.show()


In [49]:
# Visualize cluster centroids as heatmap
# centroids_z is a DataFrame.
# rows = clusters, columns = features, values = z-scores

import plotly.express as px

heatmap_features = [
    "kwh_mean", "kwh_cv", "kwh_ramp_mean", "kwh_ramp_p95",
    "v_mean", "v_std", "v_ramp_mean",
    "f_pct_outside_band",
    "i_mean", "i_p95",
    "v_i_corr", "kwh_v_corr"
]

feature_label_map = {
    "kwh_mean": "Mean Energy Consumption (kWh)",
    "kwh_cv": "Energy Variability (CV)",
    "kwh_ramp_mean": "Mean Energy Ramp Rate",
    "kwh_ramp_p95": "95th Percentile Energy Ramp",
    "v_mean": "Mean Voltage",
    "v_std": "Voltage Variability (Std)",
    "v_ramp_mean": "Mean Voltage Ramp",
    "f_pct_outside_band": "Frequency Excursions (%)",
    "i_mean": "Mean Current",
    "i_p95": "95th Percentile Current",
    "v_i_corr": "Voltage–Current Correlation",
    "kwh_v_corr": "Energy–Voltage Correlation"
}

z = centroids_z[heatmap_features]

# Map feature names to descriptive labels
x_labels = [feature_label_map[f] for f in heatmap_features]

# Compute symmetric limits around zero
z_max = float(z.abs().max().max())

fig = px.imshow(
    z,
    labels=dict(x="Feature", y="Cluster", color="Z-score"),
    x=x_labels,
    y=[f"Cluster {c}" for c in z.index],
    text_auto=".2f",
    aspect="auto",
    title="K-Means Cluster Feature Heatmap (Normalized Feature Centroids)",
    color_continuous_scale="RdBu_r",
    zmin=-z_max,
    zmax=z_max
)

fig.update_layout(template="plotly_white")
fig.update_xaxes(tickangle=-45)
fig.show()



## Figure: Cluster Heatmap (Normalized Feature Centroids) 
Normalized cluster centroids across key load, voltage, frequency, and current features. 
Positive values (red) indicate above-average behavior relative to the meter population, while negative values (blue)indicate below-average behavior. Each row represents a distinct behavioral fingerprint.

Normalized feature centroids reveal distinct behavioral patterns across clusters.
Cluster 0 represents stable baseline meters,
Cluster 1 captures demand-driven variability,
Cluster 2 highlights voltage and frequency instability consistent with upstream grid stress,
and Cluster 3 reflects high-load, high-ramping meters that amplify system stress.
